## **Aim**
To implement a program that analyzes Windows Event Log data to identify repeated failed login attempts.

## **Algorithm**
**Step 1:** Import `xml.etree.ElementTree`, `collections.Counter`, `datetime`, and `re` libraries.

**Step 2:** Create a simulated Windows Security Event Log in XML format (Event ID 4625 for failed logon, 4624 for successful).

**Step 3:** Parse the XML log and extract Event ID 4625 events (failed logon attempts).

**Step 4:** For each failed logon, extract: timestamp, target username, source IP address, logon type, failure reason (substatus code).

**Step 5:** Group failed attempts by source IP and target username.

**Step 6:** Define thresholds for brute-force detection (e.g., >5 failures from same IP in 15 minutes).

**Step 7:** Generate a report showing potential brute-force attacks and suspicious patterns.

In [1]:
import xml.etree.ElementTree as ET
from collections import defaultdict, Counter
from datetime import datetime, timedelta
import os

NAMESPACE = {'win': 'http://schemas.microsoft.com/win/2004/08/events/event'}

def create_sample_event_log(xml_file):
    """Create a simulated Windows Security Event Log XML"""
    root = ET.Element("Events")
    
    # Event IDs: 4625 = Failed logon, 4624 = Successful logon
    # Logon Types: 2=Interactive, 3=Network, 4=Batch, 5=Service, 7=Unlock, 10=RemoteInteractive
    
    events_data = [
        # Normal failed logons (user typos)
        (4625, "2026-08-20T08:00:12", "user1", "192.168.1.50", 2, "0xC000006D"),  # Bad password
        (4625, "2026-08-20T08:00:45", "user1", "192.168.1.50", 2, "0xC000006D"),
        (4625, "2026-08-20T08:01:20", "user1", "192.168.1.50", 2, "0xC000006D"),
        (4624, "2026-08-20T08:02:00", "user1", "192.168.1.50", 2, "0x0"),  # Success
        
        # Brute force from 10.0.0.100 - targeting admin
        (4625, "2026-08-20T09:15:00", "administrator", "10.0.0.100", 10, "0xC000006D"),
        (4625, "2026-08-20T09:15:05", "administrator", "10.0.0.100", 10, "0xC000006D"),
        (4625, "2026-08-20T09:15:10", "admin", "10.0.0.100", 10, "0xC000006D"),
        (4625, "2026-08-20T09:15:15", "root", "10.0.0.100", 10, "0xC000006D"),
        (4625, "2026-08-20T09:15:20", "administrator", "10.0.0.100", 10, "0xC000006D"),
        (4625, "2026-08-20T09:15:25", "admin", "10.0.0.100", 10, "0xC000006D"),
        (4625, "2026-08-20T09:15:30", "administrator", "10.0.0.100", 10, "0xC000006D"),
        (4625, "2026-08-20T09:15:35", "test", "10.0.0.100", 10, "0xC000006D"),
        (4625, "2026-08-20T09:15:40", "user", "10.0.0.100", 10, "0xC000006D"),
        
        # Another brute force from 172.16.0.50 - password spraying
        (4625, "2026-08-20T10:00:00", "john", "172.16.0.50", 3, "0xC000006D"),
        (4625, "2026-08-20T10:01:00", "jane", "172.16.0.50", 3, "0xC000006D"),
        (4625, "2026-08-20T10:02:00", "bob", "172.16.0.50", 3, "0xC000006D"),
        (4625, "2026-08-20T10:03:00", "alice", "172.16.0.50", 3, "0xC000006D"),
        (4625, "2026-08-20T10:04:00", "mike", "172.16.0.50", 3, "0xC000006D"),
        (4625, "2026-08-20T10:05:00", "sarah", "172.16.0.50", 3, "0xC000006D"),
        
        # Account lockout (Event ID 4740)
        (4740, "2026-08-20T09:15:45", "administrator", "10.0.0.100", 10, "0x0"),
        
        # Legitimate remote access
        (4624, "2026-08-20T11:00:00", "remote_user", "192.168.1.200", 10, "0x0"),
        (4624, "2026-08-20T11:30:00", "svc_backup", "192.168.1.10", 5, "0x0"),
    ]
    
    for i, (eid, ts, user, ip, logon_type, substatus) in enumerate(events_data, 1):
        event = ET.SubElement(root, "Event")
        system = ET.SubElement(event, "System")
        ET.SubElement(system, "EventID").text = str(eid)
        ET.SubElement(system, "TimeCreated", SystemTime=ts)
        ET.SubElement(system, "Computer").text = "DC01.corp.local"
        
        event_data = ET.SubElement(event, "EventData")
        ET.SubElement(event_data, "Data", Name="TargetUserName").text = user
        ET.SubElement(event_data, "Data", Name="IpAddress").text = ip
        ET.SubElement(event_data, "Data", Name="LogonType").text = str(logon_type)
        ET.SubElement(event_data, "Data", Name="SubStatus").text = substatus
        ET.SubElement(event_data, "Data", Name="WorkstationName").text = f"WKST-{ip.replace('.', '-')}"
    
    tree = ET.ElementTree(root)
    tree.write(xml_file, encoding='utf-8', xml_declaration=True)

def parse_failed_logons(xml_file):
    tree = ET.parse(xml_file)
    root = tree.getroot()
    
    failed_logons = []
    
    for event in root.findall("Event"):
        system = event.find("System")
        event_id = int(system.find("EventID").text)
        
        if event_id != 4625:
            continue
        
        timestamp_str = system.find("TimeCreated").get("SystemTime")
        timestamp = datetime.fromisoformat(timestamp_str)
        
        event_data = event.find("EventData")
        target_user = event_data.find("Data[@Name='TargetUserName']").text
        ip_addr = event_data.find("Data[@Name='IpAddress']").text
        logon_type = int(event_data.find("Data[@Name='LogonType']").text)
        substatus = event_data.find("Data[@Name='SubStatus']").text
        workstation = event_data.find("Data[@Name='WorkstationName']").text
        
        failed_logons.append({
            "timestamp": timestamp,
            "username": target_user,
            "source_ip": ip_addr,
            "logon_type": logon_type,
            "substatus": substatus,
            "workstation": workstation
        })
    
    return failed_logons

def detect_brute_force(failed_logons, window_minutes=15, threshold=5):
    """Detect brute force by IP and by username"""
    
    # Group by IP
    by_ip = defaultdict(list)
    for fl in failed_logons:
        by_ip[fl["source_ip"]].append(fl)
    
    # Group by username
    by_user = defaultdict(list)
    for fl in failed_logons:
        by_user[fl["username"]].append(fl)
    
    alerts = []
    
    # Check each IP for rapid failures
    for ip, attempts in by_ip.items():
        attempts.sort(key=lambda x: x["timestamp"])
        for i in range(len(attempts)):
            window_start = attempts[i]["timestamp"]
            window_end = window_start + timedelta(minutes=window_minutes)
            window_attempts = [a for a in attempts if window_start <= a["timestamp"] <= window_end]
            
            if len(window_attempts) >= threshold:
                unique_users = set(a["username"] for a in window_attempts)
                alerts.append({
                    "type": "BRUTE_FORCE_BY_IP",
                    "source_ip": ip,
                    "time_window": f"{window_start} to {window_end}",
                    "attempt_count": len(window_attempts),
                    "targeted_users": list(unique_users),
                    "severity": "HIGH" if len(window_attempts) >= 10 else "MEDIUM"
                })
                break  # Only report first window per IP
    
    # Check for password spraying (many users, same IP)
    for ip, attempts in by_ip.items():
        unique_users = set(a["username"] for a in attempts)
        if len(unique_users) >= 5 and len(attempts) >= 5:
            alerts.append({
                "type": "PASSWORD_SPRAYING",
                "source_ip": ip,
                "unique_users_targeted": len(unique_users),
                "total_attempts": len(attempts),
                "users": list(unique_users),
                "severity": "HIGH"
            })
    
    # Check for account enumeration (many different usernames from same IP)
    for ip, attempts in by_ip.items():
        users = Counter(a["username"] for a in attempts)
        if len(users) >= 8:
            alerts.append({
                "type": "ACCOUNT_ENUMERATION",
                "source_ip": ip,
                "usernames_tried": list(users.keys()),
                "severity": "MEDIUM"
            })
    
    return alerts

def main():
    xml_file = "security_event_log.xml"
    create_sample_event_log(xml_file)
    
    print("Parsing Windows Security Event Log...")
    failed_logons = parse_failed_logons(xml_file)
    print(f"Total failed logon events (4625): {len(failed_logons)}")
    
    alerts = detect_brute_force(failed_logons)
    
    print(f"\n{'='*70}")
    print(f"WINDOWS FAILED LOGON ANALYSIS")
    print(f"{'='*70}")
    
    # Summary by IP
    ip_counter = Counter(fl["source_ip"] for fl in failed_logons)
    print(f"\n--- Failed Logons by Source IP ---")
    for ip, count in ip_counter.most_common():
        print(f"  {ip:<20} {count} failures")
    
    # Summary by Username
    user_counter = Counter(fl["username"] for fl in failed_logons)
    print(f"\n--- Failed Logons by Target Username ---")
    for user, count in user_counter.most_common():
        print(f"  {user:<20} {count} failures")
    
    # Logon Type distribution
    type_counter = Counter(fl["logon_type"] for fl in failed_logons)
    print(f"\n--- Logon Types ---")
    type_names = {2: "Interactive", 3: "Network", 4: "Batch", 5: "Service", 7: "Unlock", 10: "RemoteInteractive"}
    for ltype, count in type_counter.most_common():
        print(f"  Type {ltype} ({type_names.get(ltype, 'Unknown')}): {count}")
    
    # Alerts
    print(f"\n--- SECURITY ALERTS ---")
    if not alerts:
        print("  No suspicious patterns detected.")
    else:
        for i, alert in enumerate(alerts, 1):
            print(f"\n  ALERT #{i} [{alert['severity']}] {alert['type']}")
            for k, v in alert.items():
                if k not in ('type', 'severity'):
                    if isinstance(v, list):
                        print(f"    {k}: {', '.join(v)}")
                    else:
                        print(f"    {k}: {v}")
    
    # Detailed failed logons
    print(f"\n--- DETAILED FAILED LOGONS ---")
    print(f"{'Time':<20} {'Username':<15} {'Source IP':<15} {'Type':<4} {'SubStatus':<12} {'Workstation'}")
    print("-" * 95)
    for fl in failed_logons:
        print(f"{fl['timestamp'].strftime('%Y-%m-%d %H:%M:%S'):<20} {fl['username']:<15} {fl['source_ip']:<15} {fl['logon_type']:<4} {fl['substatus']:<12} {fl['workstation']}")

if __name__ == "__main__":
    main()

Parsing Windows Security Event Log...
Total failed logon events (4625): 18

WINDOWS FAILED LOGON ANALYSIS

--- Failed Logons by Source IP ---
  10.0.0.100           9 failures
  172.16.0.50          6 failures
  192.168.1.50         3 failures

--- Failed Logons by Target Username ---
  administrator        4 failures
  user1                3 failures
  admin                2 failures
  root                 1 failures
  test                 1 failures
  user                 1 failures
  john                 1 failures
  jane                 1 failures
  bob                  1 failures
  alice                1 failures
  mike                 1 failures
  sarah                1 failures

--- Logon Types ---
  Type 10 (RemoteInteractive): 9
  Type 3 (Network): 6
  Type 2 (Interactive): 3

--- SECURITY ALERTS ---

  ALERT #1 [MEDIUM] BRUTE_FORCE_BY_IP
    source_ip: 10.0.0.100
    time_window: 2026-08-20 09:15:00 to 2026-08-20 09:30:00
    attempt_count: 9
    targeted_users: test, root, a

## **Result**
This the program successfully analyzes Windows Event Log data and identifies repeated failed login attempts.